# GST invoice extraction: QLoRA fine-tuning of Qwen2.5-0.5B (Colab, free T4)
Runtime → Change runtime type → **T4 GPU**, then Runtime → Run all (about 45–60 min).

This notebook reproduces every number in the README: rule-based baseline, prompted base model, fine-tuned model.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
REPO = 'https://github.com/aryanajitkadam7/gst-invoice-extractor-lora.git'
!git clone -q $REPO repo
%cd repo
!pip install -q -r requirements-gpu.txt

In [ ]:
import os; os.environ['PYTHONPATH'] = 'src'
!python -m pytest -q tests/test_data_metrics.py

## 1. Rule-based baseline (regex)

In [ ]:
!python -m invoicex.eval_rules

## 2. Base model, prompted (zero-shot and 1-shot)

In [ ]:
!python -m invoicex.infer --mode base --bs 16

## 3. QLoRA fine-tuning (4-bit NF4 base, LoRA r=16 on all attention + MLP projections)

In [ ]:
!python -m invoicex.train_lora --quant 4bit --epochs 2 --micro-bs 4 --grad-accum 4

## 4. Evaluate the fine-tuned model

In [ ]:
!python -m invoicex.infer --mode lora --bs 16

## 5. Results table + loss curve

In [ ]:
!python -m invoicex.report
import json, matplotlib.pyplot as plt
log = [json.loads(l) for l in open('reports/train_log.jsonl')]
tr = [(r['step'], r['train_loss']) for r in log if 'train_loss' in r]
va = [(r['step'], r['val_loss']) for r in log if 'val_loss' in r]
plt.plot(*zip(*tr), label='train'); plt.plot(*zip(*va), 'o-', label='val'); plt.xlabel('optimizer step'); plt.ylabel('loss'); plt.legend()
plt.savefig('reports/loss_curve.png', dpi=150); plt.show()

## 6. Save results
Download `reports.zip` and commit its contents to the repo's `reports/` folder (the adapter in `outputs/` can optionally be pushed to the Hugging Face Hub).

In [ ]:
!zip -qr reports.zip reports
from google.colab import files; files.download('reports.zip')